In [1]:
import numpy as np
from fairtfm import FairTFMClassifier, ACSPumsDataset, get_default_device, set_randomness_seed, compute_fairness_metrics
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier 


In [2]:
set_randomness_seed(42)
device = get_default_device()

dataset = ACSPumsDataset(
        acs_task="acs_income",
        states=["AL"],
        sensitive_attr_name="SEX",
        max_samples=10000,
        test_size=.2,
        seed_everything=42
    )
dataset.preprocess()
X_train, X_test, y_train, y_test, s_train, s_test = dataset.get_splits()
    
print(f"Dataset: {dataset.name}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {X_train.shape[1]} | Classes: {len(np.unique(y_train))}")
print(f"Sensitive attribute (SEX): groups {np.unique(s_train)}\n")

Dataset: acs_income_AL_SEX
Train: (3724, 9) | Test: (931, 9)
Features: 9 | Classes: 2
Sensitive attribute (SEX): groups [0 1]



In [3]:
device
models = {
        "FairTFM-0.7": FairTFMClassifier(model="checkpoints/FairTFM-0.7/checkpoint_epoch_10000.pt",device=device),
        "FairTFM-25": FairTFMClassifier(model="checkpoints/FairTFM-25/checkpoint_epoch_10000.pt",device=device),
        "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss'),
        "Random Forest": RandomForestClassifier(n_estimators=100)
    }

In [4]:
results = {}

for model_name, clf in models.items(): 

    if isinstance(clf, FairTFMClassifier): 
        clf.fit(X_train, y_train, s_train)
        y_prod = clf.predict_proba(X_test, s_test)
    else:
        clf.fit(X_train, y_train)
        y_prod = clf.predict_proba(X_test) 
    
    # ============================================================================
    # 3. FAIRNESS ANALYSIS
    # ============================================================================
    print("=" * 70)
    print(f" Fitting {model_name}")
    print("=" * 70) 
    results[model_name] = compute_fairness_metrics(y_prob=y_prod, y_test=y_test, s_test=s_test)
    

print("\n📊 PERFORMANCE METRICS:")

for model_name in results.keys():
    metrics = results[model_name]
    print(f"\n{model_name}:")
    print(f"  ROC AUC:                 {metrics['auc']:.4f}")  
    print(f"  Accuracy:                {metrics['accuracy']:.4f}")  
    print(f"  Demographic Parity Diff: {metrics['demographic_parity_difference']:.4f}")
    print(f"  Equalized Odds Diff:     {metrics['equalized_odds_difference']:.4f}")
    print(f"  Equal Opportunity Diff:  {metrics['equal_opportunity_difference']:.4f}")

 Fitting FairTFM-0.7
 Fitting FairTFM-25
 Fitting XGBoost
 Fitting Random Forest

📊 PERFORMANCE METRICS:

FairTFM-0.7:
  ROC AUC:                 0.8413
  Accuracy:                0.7658
  Demographic Parity Diff: 0.0241
  Equalized Odds Diff:     0.0335
  Equal Opportunity Diff:  0.0335

FairTFM-25:
  ROC AUC:                 0.8336
  Accuracy:                0.7068
  Demographic Parity Diff: 0.0058
  Equalized Odds Diff:     0.0092
  Equal Opportunity Diff:  0.0092

XGBoost:
  ROC AUC:                 0.8487
  Accuracy:                0.7841
  Demographic Parity Diff: 0.1226
  Equalized Odds Diff:     0.0716
  Equal Opportunity Diff:  0.0716

Random Forest:
  ROC AUC:                 0.8433
  Accuracy:                0.7895
  Demographic Parity Diff: 0.1129
  Equalized Odds Diff:     0.0823
  Equal Opportunity Diff:  0.0823
